### Experiment with using Hooke latent space for embryo projections

In [1]:
import os
import sys

sys.path.append(os.path.abspath("../../../"))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import glob2 as glob
from src.functions.plot_functions import format_3d_plotly, format_2d_plotly, rotate_figure
from pathlib import Path

# set paths
fig_root = Path("/Users/nick/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/slides/morphseq/20250528/PLN/")

fig_folder = fig_root / "cov_analyses"
fig_folder.mkdir(parents=True, exist_ok=True)

# set path to data
model_path = Path("/Users/nick/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/morphseq/results/20250312/HF_hooke_regressions/")

/Users/nick/Projects/repositories/morphseq/src/functions/__init__.py:23: UserWarning: Optional legacy import failed: src.functions.core_utils_segmentation: No module named 'segmentation_models_pytorch'
  warnings.warn(f"Optional legacy import failed: {module}: {e}")
/Users/nick/Projects/repositories/morphseq/src/functions/__init__.py:23: UserWarning: Optional legacy import failed: src.functions.spline_morph_spline_metrics: No module named 'seaborn'
  warnings.warn(f"Optional legacy import failed: {module}: {e}")
/Users/nick/Projects/repositories/morphseq/src/functions/__init__.py:23: UserWarning: Optional legacy import failed: src.functions.embryo_df_performance_metrics: No module named 'seaborn'
  warnings.warn(f"Optional legacy import failed: {module}: {e}")


In [2]:
model_path

PosixPath("/Users/nick/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/morphseq/results/20250312/HF_hooke_regressions")

### Load latent features from each temperature experiment


In [3]:
# get a list of temp model folders
folder_list = sorted(model_path.glob("*C"))

temp_vec = []
latent_list = []
col_list = []

for fname in folder_list:
    temp = int(fname.name[:-1])
    temp_vec.append(temp)

    mu = pd.read_csv(fname / "latents.csv", index_col=0)
    col_list.append(mu.columns)

    mu = mu.copy()
    mu["temperature"] = temp
    latent_list.append(mu)

common_cols = set.intersection(*map(set, col_list))
common_feature_cols = sorted(
    [col for col in common_cols if col.startswith("V")],
    key=lambda col: int(col[1:]) if col[1:].isdigit() else col,
)

latent_df = pd.concat(
    [mu.loc[:, common_feature_cols + ["temperature"]] for mu in latent_list],
    ignore_index=True,
)

print(latent_df.shape)
latent_df.head()


(144, 152)


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V143,V144,V145,V146,V147,V148,V149,V150,V151,temperature
0,-3.615554,-0.542589,0.120046,-0.565887,-3.068939,-0.998062,2.451569,0.275799,1.435192,-0.700572,...,0.344968,-1.382015,-0.174861,0.974646,-0.400257,-3.185068,-1.892851,-0.204511,2.741332,19
1,-5.039740,0.653594,-0.756586,-4.051539,-5.463499,1.692052,3.043963,-1.211869,1.617183,-1.596948,...,0.517041,-3.986889,0.039979,1.478378,-0.351259,-4.948150,-1.498824,-1.910016,2.945835,19
2,-3.516479,-0.863667,0.396414,-0.263982,-1.907309,0.330808,2.578035,0.095619,1.178153,0.717030,...,1.230266,-3.845940,-0.824413,1.715871,-3.995524,-2.305084,-1.418654,-0.315671,3.149551,19
3,-4.495378,-0.310474,0.274676,-2.573101,-4.753770,1.494272,3.233633,-1.522912,1.821750,-1.230580,...,0.754063,-4.139704,-0.896120,1.442340,-2.315526,-4.914517,-0.758920,-2.629001,2.633413,19
4,-3.189211,-0.323889,0.088325,-0.155217,-0.483296,-0.266476,2.500307,-0.447516,0.918254,-1.194725,...,0.878331,-2.131692,-0.011224,1.239733,-2.963592,-0.811265,-1.829444,0.579144,2.814629,19


## Calculate pairwise correlation between seq latent features


In [4]:
col_prefix = "V"
latent_df["temperature"].unique()


array([19, 24, 28, 32, 34, 35])

In [5]:
from tqdm import tqdm
import plotly.colors as pc
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram

n_emb_thresh = 8
feature_cols = [col for col in latent_df.columns if col.startswith(col_prefix)]
feature_cols = sorted(feature_cols, key=lambda x: int(x[1:]) if x[1:].isdigit() else x)

if len(feature_cols) == 0:
    raise ValueError(f"No columns found with prefix '{col_prefix}'")

feature_cc_fig_path = fig_folder / f"{col_prefix}_feature_cc"
feature_cc_fig_path.mkdir(parents=True, exist_ok=True)

# calculate correlation coefficients by embryo temperature
feature_temp_vec = np.asarray(sorted(latent_df["temperature"].dropna().unique())).astype(float)
feature_cc_mat_list = []
feature_cc_mat_list_null = []
feature_cov_mat_list = []
feature_count_mat_list = []

np.random.seed(371)

for temp in tqdm(feature_temp_vec):
    temp_df = latent_df.loc[latent_df["temperature"] == temp, feature_cols]
    temp_df = temp_df.apply(pd.to_numeric, errors="coerce")

    correlation_matrix = temp_df.corr(method="pearson")
    covariance_matrix = temp_df.cov()

    bool_df = temp_df.notna()
    count_matrix = bool_df.astype(int).T.dot(bool_df.astype(int))
    feature_count_mat_list.append(count_matrix.to_numpy())

    feature_cc_mat_list.append(correlation_matrix)
    feature_cov_mat_list.append(covariance_matrix)

    vals_raw = temp_df.values.flatten()
    vals = vals_raw[~np.isnan(vals_raw)]
    shuffled_values = np.random.choice(vals, len(vals_raw), replace=True).reshape(temp_df.shape)
    null_df = pd.DataFrame(shuffled_values, index=temp_df.index, columns=temp_df.columns)

    null_correlation_matrix = null_df.corr(method="pearson")
    feature_cc_mat_list_null.append(null_correlation_matrix)


100%|██████████| 6/6 [00:00<00:00, 56.16it/s]


In [6]:
# filter out features with too few paired embryos in any retained temperature group
stacked_counts = np.stack(feature_count_mat_list, axis=2)
min_count_array = np.squeeze(np.min(stacked_counts, axis=2))
feature_filter_vec = np.max(min_count_array, axis=0) >= n_emb_thresh
feature_colnames = np.asarray(feature_cc_mat_list[0].columns)

for t in range(len(feature_temp_vec)):
    # corr
    cc_temp = feature_cc_mat_list[t].copy()
    feature_cc_mat_list[t] = cc_temp.loc[feature_colnames[feature_filter_vec], feature_colnames[feature_filter_vec]]

    # null corr
    cc_temp_n = feature_cc_mat_list_null[t].copy()
    feature_cc_mat_list_null[t] = cc_temp_n.loc[feature_colnames[feature_filter_vec], feature_colnames[feature_filter_vec]]

    # cov
    cv_temp = feature_cov_mat_list[t].copy()
    feature_cov_mat_list[t] = cv_temp.loc[feature_colnames[feature_filter_vec], feature_colnames[feature_filter_vec]]

print(f"Retained {feature_filter_vec.sum()} of {len(feature_filter_vec)} {col_prefix} features")


Retained 151 of 151 V features


In [7]:
feature_temp_vec

array([19., 24., 28., 32., 34., 35.])

In [8]:
# get sort order from the 28C control group when available, otherwise use the middle temperature group
if 28 in feature_temp_vec:
    t_ind = int(np.where(feature_temp_vec == 28)[0][0])
else:
    t_ind = len(feature_temp_vec) // 2

corr = feature_cc_mat_list[t_ind].copy()
corr_for_cluster = corr.fillna(0)
np.fill_diagonal(corr_for_cluster.values, 1)

# Compute hierarchical clustering on 1 - Pearson correlation.
dist = 1 - corr_for_cluster
condensed_dist = squareform(dist, checks=False)
Z = linkage(condensed_dist, method="average")
dendro = dendrogram(Z, no_plot=True)
feature_order = dendro["leaves"]


In [13]:
# for t, temp in enumerate(feature_temp_vec):

t = 5
temp = feature_temp_vec[t]
print(temp)

cc_mat = feature_cc_mat_list[t].iloc[feature_order, feature_order]
fig = px.imshow(cc_mat, color_continuous_scale="RdBu_r", range_color=[-1, 1])

fig.update_layout(width=600, height=600)
fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    font=dict(color="white", family="Arial, sans-serif", size=18)
)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_traces(xgap=1, ygap=1)
fig.update_layout(plot_bgcolor="black", paper_bgcolor="black")
fig.update_layout(coloraxis_showscale=False)
fig.update_layout(margin=dict(l=10, r=10, t=10, b=10))
fig.update_xaxes(automargin=False)
fig.update_yaxes(automargin=False)

fig.write_image(feature_cc_fig_path / f"{col_prefix}_feature_cc_{temp:g}C.png", scale=2)
fig.write_html(feature_cc_fig_path / f"{col_prefix}_feature_cc_{temp:g}C.html")
cc_mat.to_csv(feature_cc_fig_path / f"{col_prefix}_feature_cc_{temp:g}C.csv", index=True)

fig.show()


35.0


In [10]:
feature_cc_list = []
feature_cc_null_val_list = []

for t, temp in enumerate(feature_temp_vec):
    cc = feature_cc_mat_list[t].to_numpy()
    tril_indices = np.tril_indices(cc.shape[0], k=-1)
    cc_vals = cc[tril_indices]

    cc_n = feature_cc_mat_list_null[t].to_numpy()
    cc_n_vals = cc_n[tril_indices]
    feature_cc_null_val_list.append(cc_n_vals[cc_vals <= 1])

    cc_df_temp = pd.DataFrame(cc_vals[cc_vals <= 1], columns=["cc"])
    cc_df_temp["temperature"] = temp
    feature_cc_list.append(cc_df_temp)

feature_null_df = pd.DataFrame(np.concatenate(feature_cc_null_val_list), columns=["cc"])
feature_null_df["temperature"] = "null"

feature_cc_df = pd.concat(feature_cc_list + [feature_null_df], axis=0, ignore_index=True)
feature_cc_df.head()


,cc,temperature
0,-0.540827,19.0
1,0.084827,19.0
2,0.059464,19.0
3,-0.057560,19.0
4,-0.181030,19.0


In [11]:
feature_cc_plot_df = feature_cc_df.loc[feature_cc_df["temperature"] != "null"].copy()
feature_cc_plot_df["temperature"] = feature_cc_plot_df["temperature"].astype(float)
feature_cc_plot_df["cc_abs"] = feature_cc_plot_df["cc"].abs()

feature_group_stats = feature_cc_plot_df.groupby(["temperature"])["cc_abs"].agg(
    mean="mean",
    std="std",
    sem="sem"
).reset_index()

groups = feature_cc_plot_df["temperature"].unique()
min_val, max_val = 17, 38
colors = dict({float(val):
    pc.sample_colorscale("RdBu_r", (val - min_val) / (max_val - min_val))[0]
    for val in groups
})

feature_cc_plot_df["temperature_group"] = pd.Categorical(feature_cc_plot_df["temperature"].astype(str))
feature_cc_plot_df = feature_cc_plot_df.sort_values(["temperature_group"])

fig = px.box(feature_cc_plot_df, y="cc_abs", x="temperature", color="temperature", color_discrete_map=colors)
fig.update_traces(width=0.8)
fig = format_2d_plotly(fig, font_size=20, axis_labels=["temperature (C)", "correlation coefficient"])
fig.update_traces(opacity=0.95)
fig.update_traces(line=dict(width=5))

fig.show()

fig.write_image(feature_cc_fig_path / f"{col_prefix}_feature_cc_box.png", scale=2)
fig.write_html(feature_cc_fig_path / f"{col_prefix}_feature_cc_box.html")


In [12]:
fig = px.scatter(
    feature_group_stats,
    x="temperature",
    y="mean",
    error_y="sem",
    color="temperature",
    color_continuous_scale="RdBu_r",
    range_color=[17, 38]
)

fig = format_2d_plotly(fig, axis_labels=["temperature (C)", "correlation coefficient"], font_size=20, marker_size=24)
fig.update_traces(mode="lines+markers", line=dict(color="white"))
# fig.update_layout(yaxis=dict(range=[0, 0.3]))
fig.update_layout(coloraxis_showscale=False)
fig.update_traces(error_x=dict(color="white", width=0))

fig.show()

fig.write_image(feature_cc_fig_path / f"{col_prefix}_feature_cc_scatter.png", scale=2)
fig.write_html(feature_cc_fig_path / f"{col_prefix}_feature_cc_scatter.html")


In [14]:
import numpy as np
import pandas as pd
import plotly.express as px

# -----------------------------
# Metric functions
# -----------------------------

def _offdiag_values(R):
    R = np.asarray(R, dtype=float)
    mask = ~np.eye(R.shape[0], dtype=bool)
    return R[mask]

def mean_abs_offdiag_corr(R):
    vals = _offdiag_values(R)
    return np.nanmean(np.abs(vals))

def rms_offdiag_corr(R):
    vals = _offdiag_values(R)
    return np.sqrt(np.nanmean(vals ** 2))

def effective_rank(R, eps=1e-12):
    R = np.asarray(R, dtype=float)

    # Replace NaNs if needed
    R = np.nan_to_num(R, nan=0.0)

    eigvals = np.linalg.eigvalsh(R)
    eigvals = np.clip(eigvals, 0, None)

    if eigvals.sum() <= eps:
        return np.nan

    q = eigvals / eigvals.sum()
    entropy = -np.sum(q * np.log(q + eps))
    return np.exp(entropy)

def top_k_eigen_fraction(R, k=3, eps=1e-12):
    R = np.asarray(R, dtype=float)
    R = np.nan_to_num(R, nan=0.0)

    eigvals = np.linalg.eigvalsh(R)
    eigvals = np.clip(eigvals, 0, None)
    eigvals = np.sort(eigvals)[::-1]

    if eigvals.sum() <= eps:
        return np.nan

    return eigvals[:k].sum() / eigvals.sum()


In [15]:
# -----------------------------
# Calculate metrics
# -----------------------------

metric_records = []

for t, temp in enumerate(feature_temp_vec):
    cc_mat = feature_cc_mat_list[t].iloc[feature_order, feature_order]

    metric_records.append({
        "temperature": float(temp),
        "mean_abs_offdiag_corr": mean_abs_offdiag_corr(cc_mat),
        "rms_offdiag_corr": rms_offdiag_corr(cc_mat),
        "effective_rank": effective_rank(cc_mat),
        "top1_eigen_fraction": top_k_eigen_fraction(cc_mat, k=1),
        "top3_eigen_fraction": top_k_eigen_fraction(cc_mat, k=3),
        "top5_eigen_fraction": top_k_eigen_fraction(cc_mat, k=5),
    })

feature_order_metric_df = pd.DataFrame(metric_records).sort_values("temperature")
feature_order_metric_df.to_csv(feature_cc_fig_path / f"{col_prefix}_feature_order_metrics.csv", index=False)


# -----------------------------
# Plot helper
# -----------------------------

def plot_metric(metric, y_label=None):
    if y_label is None:
        y_label = metric.replace("_", " ")

    fig = px.scatter(
        feature_order_metric_df,
        x="temperature",
        y=metric,
        color="temperature",
        color_continuous_scale="RdBu_r",
        range_color=[17, 38],
    )

    fig = format_2d_plotly(
        fig,
        axis_labels=["temperature (C)", y_label],
        font_size=20,
        marker_size=24,
    )

    fig.update_traces(
        mode="lines+markers",
        line=dict(color="white"),
    )

    fig.update_layout(coloraxis_showscale=False)

    fig.show()

    fig.write_image(feature_cc_fig_path / f"{col_prefix}_{metric}.png", scale=2)
    fig.write_html(feature_cc_fig_path / f"{col_prefix}_{metric}.html")

    return fig


# -----------------------------
# Make plots
# -----------------------------

plot_metric("mean_abs_offdiag_corr", "mean |correlation coefficient|")
plot_metric("rms_offdiag_corr", "RMS off-diagonal correlation")
plot_metric("effective_rank", "effective rank")
plot_metric("top1_eigen_fraction", "top 1 eigenvalue fraction")
plot_metric("top3_eigen_fraction", "top 3 eigenvalue fraction")
plot_metric("top5_eigen_fraction", "top 5 eigenvalue fraction")

feature_order_metric_df


,temperature,mean_abs_offdiag_corr,rms_offdiag_corr,effective_rank,top1_eigen_fraction,top3_eigen_fraction,top5_eigen_fraction
0,19.0,0.307913,0.381316,9.913458,0.299545,0.572152,0.696118
1,24.0,0.337788,0.402416,9.998433,0.345406,0.601501,0.706697
2,28.0,0.353718,0.421292,9.053675,0.328400,0.636055,0.727713
3,32.0,0.374422,0.438992,8.707459,0.362959,0.656012,0.739912
4,34.0,0.355355,0.416894,9.589063,0.353938,0.606814,0.710187
5,35.0,0.404679,0.467522,8.375038,0.436062,0.641070,0.733887


In [ ]:
from dev.seq.hooke_latent_projections.project_ccs_data import construct_X
from tqdm import tqdm 

# dis = 2.0
expt = "NA" #"expthotfish2"
cov_col_list = beta_array.columns.tolist()

# gene3_dict = dict({"expt":"exptGENE3"})
null_dict = dict({"expt":"NA"})
# generate covariate matrix

nt = 100
time_ref_vals = np.linspace(np.min(metadata_df["timepoint"]), np.max(metadata_df["timepoint"]), nt)
# time_pd_vals = time_df["pseudostage"].to_numpy()

# construct covariates matrix
x_list = []
for t in tqdm(time_ref_vals):
    xt = construct_X(timepoint=t, cov_dict=null_dict, cov_col_list=cov_col_list, spline_lookup_df=spline_lookup_df)
    x_list.append(xt)

# x_list_pd = []
# for t in tqdm(time_pd_vals):
#     xt = construct_X(timepoint=t, cov_dict=null_dict, cov_col_list=cov_col_list, spline_lookup_df=spline_lookup_df)
#     x_list_pd.append(xt)


# # get covariate array
X = pd.concat(x_list)
# X_pd = pd.concat(x_list_pd)
# # X3 = pd.concat(x_list3)

# # get prediction matrix
ref_latent_df = (X @ beta_array.T).reset_index(drop=True)
# pd_latent_df = (X_pd @ beta_array.T)

# residual_df = pd.DataFrame(latents_df.to_numpy() - pd_latent_array.to_numpy(), columns=pd_latent_array.columns)
# # ref_latent_array3 = X3 @ beta_array.T
ref_latent_df.head()

In [ ]:
from sklearn.decomposition import PCA
n_components = 25

pca_cols = [f"PCA_{p:02}" for p in range(n_components)]

pca = PCA(n_components=n_components)
pca.fit(latents_df)

# get loadings
ccs_pca = pd.DataFrame(pca.transform(latents_df), columns=pca_cols, index=latents_df.index)
ref_pca = pd.DataFrame(pca.transform(ref_latent_df), columns=pca_cols, index=ref_latent_df.index)

# plot % explained
fig = px.line(x=np.arange(n_components) + 1, y=np.cumsum(pca.explained_variance_ratio_), markers=True)

fig.update_layout(
    title = "PC loadings",
    xaxis_title="PC",
    yaxis_title="% variance explained")

fig.show()

# fig.write_image(fig_path + "pc_var.png", scale=2)
# fig.write_html(fig_path + "pc_var.html")

#### Plot wildtype curve

In [ ]:
# ccs_pca = ccs_pca.rename(columns={"temp":"temperature"})
ccs_pca["inference_flag"] = time_df["inference_flag"].copy()
ccs_pca["timepoint"] = time_df["timepoint"].copy()
ccs_pca["temperature"] = time_df["temp"].copy()
ccs_pca["expt"] = time_df["expt"].copy()
ccs_pca["pseudostage"] = time_df["pseudostage"].copy()

# general perspective params
zoom_factor = 0.021
z_rotation = 25
elevation = -10
marker_size = 6

xrange = [-25, 25]
yrange = [-12, 30]
zrange = [-11, 15]

# plot params
ref_filter = ccs_pca.loc[:, "inference_flag"]==1
colormap = "RdBu_r"
color_range = [17, 39]

axis_labels = ["seq PC 1", "seq PC 2", "seq PC 3"]

fig = px.scatter_3d(ccs_pca.loc[ref_filter], x=pca_cols[0], y=pca_cols[1], z=pca_cols[2],
                    color="timepoint", opacity=1)

fig.layout.scene.xaxis.range = xrange
fig.layout.scene.yaxis.range = yrange
fig.layout.scene.zaxis.range = zrange

fig.update_traces(marker=dict(size=5))
fig.add_traces(go.Scatter3d(x=ref_pca.loc[:, pca_cols[0]], y=ref_pca.loc[:, pca_cols[1]], z=ref_pca.loc[:, pca_cols[2]], 
                             mode="lines", line=dict(color="white", width=4), showlegend=False)) # marker=dict(size=3),

fig = format_3d_plotly(fig, axis_labels=axis_labels, marker_size=marker_size, font_size=18)

fig = rotate_figure(fig, zoom_factor=zoom_factor, z_rotation=z_rotation, elev_rotation=elevation)


# fig.add_traces(go.Scatter3d(x=ref3_pca[:, 0], y=ref3_pca[:, 1], z=ref3_pca[:, 2]))#, 
                            # marker=dict(color=time_vals, size=3), line=dict(color="black")))

fig.show()

fig.write_image(fig_path + "hf2_ref_pca_trajectory.png", scale=2)
fig.write_html(fig_path + "hf2_ref_pca_trajectory.html")

In [ ]:

tmin = np.min(time_ref_vals)
tmax = np.max(time_ref_vals)
time_vec = np.linspace(tmin, tmax, 50)

spline_frame_path = os.path.join(fig_path, "hf_seq_spline_pca_frames", "")
os.makedirs(spline_frame_path, exist_ok=True)


# rotate
for t, time in enumerate(tqdm(time_vec)):

# t = 25
# time = time_vec[t]

# if True:
    # initialize figure
    fig = px.scatter_3d(ccs_pca.loc[ref_filter], x=pca_cols[0], y=pca_cols[1], z=pca_cols[2],
                        color="timepoint", opacity=1)
    
    fig.layout.scene.xaxis.range = xrange
    fig.layout.scene.yaxis.range = yrange
    fig.layout.scene.zaxis.range = zrange


    time_filter = time_ref_vals <= time
    last_true_only = np.arange(len(time_filter)) == np.nonzero(time_filter)[0][-1]
    
    fig.add_traces(go.Scatter3d(x=ref_pca.loc[time_filter, pca_cols[0]], y=ref_pca.loc[time_filter, pca_cols[1]], 
                                z=ref_pca.loc[time_filter, pca_cols[2]], 
                                 mode="lines", line=dict(color="white", width=4), showlegend=False)) 

    # fig.add_traces(go.Scatter3d(x=ref_pca.loc[time_filter, pca_cols[0]], y=ref_pca.loc[time_filter, pca_cols[1]], 
    #                             z=ref_pca.loc[time_filter, pca_cols[2]], 
    #                              mode="markers", marker=dict(color="white", size=20), showlegend=False)) 
    
    fig = format_3d_plotly(fig, axis_labels=axis_labels, marker_size=marker_size, font_size=18)

    if t > 0:
        fig.add_traces(go.Scatter3d(x=ref_pca.loc[last_true_only, pca_cols[0]].values, 
                                    y=ref_pca.loc[last_true_only, pca_cols[1]].values, 
                                    z=ref_pca.loc[last_true_only, pca_cols[2]].values, 
                                    mode="markers", marker=dict(color="white", size=9, line=dict(color="black", width=1)), 
                                    showlegend=False)) 

    fig = rotate_figure(fig, zoom_factor=zoom_factor, z_rotation=z_rotation, elev_rotation=elevation)
    

    fig.write_image(os.path.join(spline_frame_path, f"hotfish_seq_pca_time{t:02}.png"), scale=2)
    
fig.show()

In [ ]:
angle_vec = np.linspace(0, 360, 50)

frame_path = os.path.join(fig_path, "hf_ref_seq_pca_rot_frames", "")
os.makedirs(frame_path, exist_ok=True)

# initialize figure
fig = px.scatter_3d(ccs_pca.loc[ref_filter], x=pca_cols[0], y=pca_cols[1], z=pca_cols[2],
                    color="timepoint", opacity=1)

fig.layout.scene.xaxis.range = xrange
fig.layout.scene.yaxis.range = yrange
fig.layout.scene.zaxis.range = zrange

fig.update_traces(marker=dict(size=5))
fig.add_traces(go.Scatter3d(x=ref_pca.loc[:, pca_cols[0]], y=ref_pca.loc[:, pca_cols[1]], z=ref_pca.loc[:, pca_cols[2]], 
                             mode="lines", line=dict(color="white", width=4), showlegend=False)) # marker=dict(size=3),

fig = format_3d_plotly(fig, axis_labels=axis_labels, marker_size=marker_size, font_size=18)


# rotate
for a, angle in enumerate(tqdm(angle_vec)):
    
    
    fig = rotate_figure(fig, zoom_factor=zoom_factor, z_rotation=z_rotation + angle, elev_rotation=elevation)
    

    fig.write_image(os.path.join(frame_path, f"hotfish_seq_pca_rot{a:02}.png"), scale=2)
    # fig.write_html(os.path.join(fig_path, f"hotfish_pca_rot{a:02}.html"))
    
fig.show()

In [ ]:
hot_filter = ccs_pca.loc[:, "expt"]=='hotfish2'

hf_frame_path = os.path.join(fig_path, "hf_seq_pca_scatter_frames", "")
os.makedirs(hf_frame_path, exist_ok=True)

ccs_stage_vec = ccs_pca.loc[:, "pseudostage"].to_numpy()
smin = np.min(ccs_pca.loc[hot_filter, "pseudostage"]) - 1
smax = np.max(ccs_pca.loc[hot_filter, "pseudostage"])

stage_vec = np.linspace(smin, smax, 25)

for s, stage in enumerate(tqdm(stage_vec)):

    if s > 0:
        opacity = 1
        stage_filter = ccs_stage_vec <= stage
    else:
        opacity = 0
        stage_filter = ccs_stage_vec <= np.inf

    plot_filter = stage_filter & hot_filter
    

    
    # initialize figure
    fig = px.scatter_3d(ccs_pca.loc[plot_filter], x=pca_cols[0], y=pca_cols[1], z=pca_cols[2], opacity=opacity,
                        color="temperature", color_continuous_scale=colormap, range_color=color_range)
    
    fig.layout.scene.xaxis.range = xrange
    fig.layout.scene.yaxis.range = yrange
    fig.layout.scene.zaxis.range = zrange

    
    fig.add_traces(go.Scatter3d(x=ref_pca.loc[:, pca_cols[0]], y=ref_pca.loc[:, pca_cols[1]], 
                                z=ref_pca.loc[:, pca_cols[2]], 
                                 mode="lines", line=dict(color="white", width=4), showlegend=False)) 

    # fig.add_traces(go.Scatter3d(x=ref_pca.loc[time_filter, pca_cols[0]], y=ref_pca.loc[time_filter, pca_cols[1]], 
    #                             z=ref_pca.loc[time_filter, pca_cols[2]], 
    #                              mode="markers", marker=dict(color="white", size=20), showlegend=False)) 
    
    fig = format_3d_plotly(fig, axis_labels=axis_labels, marker_size=marker_size, font_size=18)

    fig = rotate_figure(fig, zoom_factor=zoom_factor, z_rotation=z_rotation, elev_rotation=elevation)
    

    fig.write_image(os.path.join(hf_frame_path, f"hotfish_seq_pca_stage{s:02}.png"), scale=2)
    
fig.show()

### Look at inferred pseudotime vs experimental timepoint

In [ ]:
time_df["temperature"] = time_df["temp"].copy()

fig = px.scatter(time_df.loc[hot_filter], x="mean_nn_time", y="pseudostage", color="temperature", symbol="timepoint",
                color_continuous_scale=colormap, range_color=color_range)

# Compute a linear trendline on all data
x = time_df.loc[hot_filter, "mean_nn_time"] 
y = time_df.loc[hot_filter, "pseudostage"] 
coeffs = np.polyfit(x, y, 1)  # Linear fit (degree 1)
poly = np.poly1d(coeffs)

# Create a smooth line for the trendline
x_line = np.linspace(x.min()-2, x.max()+2, 100)
y_line = poly(x_line)

# Add the overall trendline as a trace
fig.add_trace(
    go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines", showlegend=False,
        line=dict(color="white", width=2.5, dash="dash")
    )
)

fig.layout.xaxis.range = [18, 45]
fig.layout.yaxis.range = [18, 45]


axis_labels = ["nn transcritpional age (hpf)", "pseudostage (hpf)"]
fig = format_2d_plotly(fig, axis_labels=axis_labels, font_size=20, marker_size=10)

# fig.update_coloraxes(colorbar_title="temperature")
fig.show()

fig.write_image(fig_path + "hf2_pseudotime_vs_nn_time.png", scale=2)
fig.write_html(fig_path + "hf2_pseudotime_vs_nn_time.html")

### Plot vs expected

In [ ]:
# now group by cohort
time_df["stage_shift"] = time_df["pseudostage"] - time_df["timepoint"] 
cohort_stage_df = time_df.loc[:, ["temperature", "timepoint", "expt", "mean_nn_time", "pseudostage", "stage_shift"]].groupby(
                    ["temperature", "timepoint", "expt"]).agg(["mean", "std"])

cohort_stage_df.columns = [f"{col[0]}_{col[1]}" for col in cohort_stage_df.columns]
cohort_stage_df = cohort_stage_df.reset_index()

hot_filter = cohort_stage_df["expt"]=="hotfish2"
# get predicted stage using linear formula
cohort_stage_df["predicted_stage"] = 6 + (cohort_stage_df["timepoint"]-6)*(0.055*cohort_stage_df["temperature"]-0.57)

ref_vec = np.linspace(14, 44)
marker_size = 14


fig = px.scatter(cohort_stage_df.loc[hot_filter], x="predicted_stage", y="pseudostage_mean", error_y="pseudostage_std", 
                 color="temperature", symbol="timepoint",color_continuous_scale=colormap, range_color=[19, 35])

fig.update_traces(error_y=dict(width=0))
# fig.update_traces(mode="lines+markers", line=dict(color="white", width=0.5))

fig.add_trace(go.Scatter(x=ref_vec, y=ref_vec, mode="lines", line=dict(color="white", width=2.5, dash="dash"), showlegend=False))

axis_labels = ["expected stage (hpf)", "molecular staging <br> (nn-transcriptional age)"]

fig = format_2d_plotly(fig, marker_size=marker_size, axis_labels=axis_labels, font_size=20)#, show_gridlines=False)


fig.show()

fig.write_image(fig_path + "cohort_pseudotime_seq_stage_all.png", scale=2)
fig.write_html(fig_path + "cohort_pseudotime_seq_stage_all.html")

In [ ]:
fig = px.scatter(cohort_stage_df.loc[hot_filter], x="mean_nn_time_mean", y="pseudostage_mean", 
                 error_y="pseudostage_std", error_x="mean_nn_time_std",
                 color="temperature", symbol="timepoint",color_continuous_scale=colormap, range_color=[19, 35])

fig.update_traces(error_y=dict(width=0))
fig.update_traces(error_x=dict(width=0))

# fig.update_traces(mode="lines+markers", line=dict(color="white", width=0.5))

fig.add_trace(go.Scatter(x=ref_vec, y=ref_vec, mode="lines", line=dict(color="white", width=2.5, dash="dash"), showlegend=False))

axis_labels = ["nn-transcriptional age", "pseudotime"]

fig = format_2d_plotly(fig, marker_size=marker_size, axis_labels=axis_labels, font_size=20)#, show_gridlines=False)


fig.show()

fig.write_image(fig_path + "mean_pseudotime_vs_nn.png", scale=2)
fig.write_html(fig_path + "mean_pseudotime_vs_nn.html")

In [ ]:
cohort_stage_df["pseudostage_cv"] = np.divide(cohort_stage_df["pseudostage_std"], cohort_stage_df["pseudostage_mean"])

fig = px.scatter(cohort_stage_df.loc[hot_filter], x="stage_shift_mean", y="pseudostage_cv", 
                 color="temperature", symbol="timepoint",color_continuous_scale=colormap, range_color=[19, 35])

fig.update_traces(error_y=dict(width=0))
# fig.update_traces(mode="lines+markers", line=dict(color="white", width=0.5))

# fig.add_trace(go.Scatter(x=ref_vec, y=ref_vec, mode="lines", line=dict(color="white", width=2.5, dash="dash"), showlegend=False))

axis_labels = ["stage shift", "stage dispersion"]

fig = format_2d_plotly(fig, marker_size=marker_size, axis_labels=axis_labels, font_size=20)#, show_gridlines=False)


fig.show()

fig.write_image(fig_path + "cohort_shift_noise_seq.png", scale=2)
fig.write_html(fig_path + "cohort_shift_noise_seq.html")

### Look to see if we see temp-shift in hotfish experiment

In [ ]:
hot_filter = time_df["expt"]=='hotfish2'
# time_df["expt"].unique()

fig = px.scatter(time_df.loc[hot_filter], x="timepoint", y="pseudostage", color="temp")

# fig.update_layout(xaxis=dict(range=[0, 120]), 
#                   yaxis=dict(range=[0, 120]))
fig.update_layout(width=800, height=600) 

fig.update_layout(
    title = "predicted stage vs experiment time (hotfish)",
    xaxis_title="experimental timepoint (hpf)",
    yaxis_title="transcriptional pseudo-stage (hpf)")

fig.update_coloraxes(colorbar_title="temperature (C)")
fig.show()

fig.write_image(fig_path + "hf2_pseudotime_vs_timepoint.png", scale=2)
fig.write_html(fig_path + "hf2_pseudotime_vs_timepoint.html")

# fig.show()

In [ ]:
fig = px.scatter(time_df.loc[hot_filter], x="mean_nn_time", y="pseudostage", color="temp")

# fig.update_layout(xaxis=dict(range=[0, 120]), 
#                   yaxis=dict(range=[0, 120]))
fig.update_layout(width=800, height=600) 

fig.update_layout(
    title = "predicted stage vs experiment time (hotfish)",
    xaxis_title="nearest-neighbor stage (hpf)",
    yaxis_title="transcriptional pseudo-stage (hpf)")

fig.update_coloraxes(colorbar_title="temperature (C)")
fig.show()

fig.write_image(fig_path + "hf2_pseudotime_vs_nn_time.png", scale=2)
fig.write_html(fig_path + "hf2_pseudotime_vs_nn_time.html")

### Let's use PCA to visualize latent space

In [ ]:


# get mean model predictions
hooke_data_path = "/Users/nick/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/morphseq/seq_data/emb_projections/hooke_model_files/"
model_path = os.path.join(hooke_data_path, model_name, "")

# load spline lookup
spline_lookup_df = pd.read_csv(model_path + "time_splines.csv")

# load hooke model files
cov_array = pd.read_csv(model_path + "COV.csv", index_col=0)
beta_array = pd.read_csv(model_path + "B.csv", index_col=0).T

beta_array = beta_array.rename(columns={"(Intercept)":"Intercept"})

cols_from = beta_array.columns
cols_from_clean = [col.replace(" = c", "=") for col in cols_from]
beta_array.columns = cols_from_clean
beta_array.head()

In [ ]:
from sklearn.decomposition import PCA
n_components = 25
# n_cell_cutoff = 250
# n_filter = metadata_df["count_per_embryo"]>=n_cell_cutoff
# latents_df_filtered = latents_df.loc[n_filter, :]

pca = PCA(n_components=n_components)
pca.fit(residual_df)

# get loadings
ccs_pca_array = pca.transform(latents_df)
ref_pca = pca.transform(ref_latent_array)

# plot % explained
fig = px.line(x=np.arange(n_components), y=pca.explained_variance_ratio_, markers=True)

fig.update_layout(
    title = "PC loadings",
    xaxis_title="PC",
    yaxis_title="% variance explained")

fig.show()

fig.write_image(fig_path + "pc_var.png", scale=2)
fig.write_html(fig_path + "pc_var.html")

In [ ]:
from dev.seq.hooke_latent_projections.project_ccs_data import construct_X
from tqdm import tqdm 

# dis = 2.0
expt = "NA" #"expthotfish2"
cov_col_list = beta_array.columns.tolist()

# gene3_dict = dict({"expt":"exptGENE3"})
null_dict = dict({"expt":"NA"})
# generate covariate matrix
nt = 100
time_ref_vals = np.linspace(np.min(metadata_df["timepoint"]), np.max(metadata_df["timepoint"]), nt)
time_pd_vals = time_df["pseudostage"].to_numpy()

# construct covariates matrix
x_list = []
for t in tqdm(time_ref_vals):
    xt = construct_X(timepoint=t, cov_dict=null_dict, cov_col_list=cov_col_list, spline_lookup_df=spline_lookup_df)
    x_list.append(xt)

x_list_pd = []
for t in tqdm(time_pd_vals):
    xt = construct_X(timepoint=t, cov_dict=null_dict, cov_col_list=cov_col_list, spline_lookup_df=spline_lookup_df)
    x_list_pd.append(xt)


# # get covariate array
X = pd.concat(x_list)
X_pd = pd.concat(x_list_pd)
# # X3 = pd.concat(x_list3)

# # get prediction matrix
ref_latent_array = X @ beta_array.T
pd_latent_array = X_pd @ beta_array.T

residual_df = pd.DataFrame(latents_df.to_numpy() - pd_latent_array.to_numpy(), columns=pd_latent_array.columns)
# # ref_latent_array3 = X3 @ beta_array.T
ref_latent_array.head()

In [ ]:
pd_latent_array.shape
latents_df.shape

#### Seems like we have a significant QC issue with Gene3...there are ~5/10x fewer cells?!?

In [ ]:
metadata_df["log_counts"] = np.log10(metadata_df["count_per_embryo"])
fig = px.histogram(metadata_df, x="log_counts", color="expt", opacity=0.75)
# fig.update_xaxes(type="log")
fig.update_layout(barmode="overlay")
fig.show()

In [ ]:
time_df_ft = time_df.loc[n_filter]
hot_filter =time_df_ft.loc[:, "expt"]=='hotfish2'

fig = px.scatter_3d(x=ccs_pca_array[hot_filter, 0], y=ccs_pca_array[hot_filter, 1], z=ccs_pca_array[hot_filter, 2],
                    color=time_df_ft.loc[hot_filter, "temp"], opacity=0.7,
                    size=time_df_ft.loc[hot_filter, "timepoint"])
fig.update_traces(marker=dict(size=5))
fig.add_traces(go.Scatter3d(x=ref_pca[:, 0], y=ref_pca[:, 1], z=ref_pca[:, 2], 
                            marker=dict(size=3), line=dict(color="black")))
# fig.add_traces(go.Scatter3d(x=ref3_pca[:, 0], y=ref3_pca[:, 1], z=ref3_pca[:, 2]))#, 
                            # marker=dict(color=time_vals, size=3), line=dict(color="black")))

fig.show()

fig.write_image(fig_path + "hf2_hotfish_pca_trajectory.png", scale=2)
fig.write_html(fig_path + "hf2_hotfish_pca_trajectory.html")

In [ ]:
chem_filter = np.asarray([1 if "chem" in exp.lower() else 0 for exp in metadata_df["expt"].tolist()])==1
chem_i_vec = metadata_df.loc[chem_filter, "expt"]

In [ ]:
# fig = px.scatter_3d(x=ccs_pca_array[chem_filter, 0], y=ccs_pca_array[chem_filter, 1], z=ccs_pca_array[chem_filter, 2],
#                     color=chem_i_vec)
# fig.update_traces(marker=dict(size=5))
# fig.add_traces(go.Scatter3d(x=ref_pca[:, 0], y=ref_pca[:, 1], z=ref_pca[:, 2], 
#                             marker=dict(color=time_vals, size=3), line=dict(color="black")))

# fig.show()

In [ ]:
crisp_filter = np.asarray([1 if "chem" in exp.lower() else 0 for exp in metadata_df["expt"].tolist()])==1
chem_i_vec = metadata_df.loc[chem_filter, "target"]

fig = px.scatter_3d(x=ccs_pca_array[chem_filter, 0], y=ccs_pca_array[chem_filter, 1], z=ccs_pca_array[chem_filter, 2],
                    color=chem_i_vec)
fig.update_traces(marker=dict(size=5))
fig.add_traces(go.Scatter3d(x=ref_pca[:, 0], y=ref_pca[:, 1], z=ref_pca[:, 2], 
                            marker=dict(color=time_vals, size=3), line=dict(color="black")))

fig.show()

In [ ]:
# metadata_df["perturbation"].unique()
# ctrl_labels = np.asarray(["EtOH", "DMSO","ctrl-inj", "reference", "ctrl-uninj", "novehicle"])
# ctrl_filter = np.isin(metadata_df["perturbation"], ctrl_labels)
# bead_filter = (metadata_df["dis_protocol"]==2).to_numpy()
# np.sum(ctrl_filter & bead_filter)
# metadata_df.loc[metadata_df["target"]=="Control", "perturbation"].unique()

In [ ]:
metadata_df_ft = metadata_df.loc[n_filter, :]
# inf_filter = metadata_df_ft["target"]=="Control"

fig = px.scatter_3d(x=ccs_pca_array[:, 0], y=ccs_pca_array[:, 1], z=ccs_pca_array[:, 2],
                    color=metadata_df_ft.loc[:, "target"]
                   )
fig.update_traces(marker=dict(size=5))
fig.add_traces(go.Scatter3d(x=ref_pca[:, 0], y=ref_pca[:, 1], z=ref_pca[:, 2], 
                            marker=dict(color=time_vals, size=3), line=dict(color="black")))

fig.show()

In [ ]:
import umap

umap_model = umap.UMAP(n_components=3, n_neighbors=7, min_dist=1, metric='euclidean')

# Compute the embedding
embedding = umap_model.fit_transform(residual_df)
ref = umap_model.transform(ref_latent_array)

In [ ]:
fig = px.scatter_3d(x=embedding[hot_filter, 0], y=embedding[hot_filter, 1], z=embedding[hot_filter, 2],
                    color=time_df.loc[hot_filter, "pseudostage"], hover_data=[time_df.loc[hot_filter,"pseudostage"]])

fig.update_traces(marker=dict(size=5))
fig.add_traces(go.Scatter3d(x=ref[:, 0], y=ref[:, 1], z=ref[:, 2], 
                            marker=dict(color=time_vals, size=3), line=dict(color="black")))

fig.show()

fig.write_image(fig_path + "hf2_hotfish_umap_trajectory.png", scale=2)
fig.write_html(fig_path + "hf2_hotfish_umap_trajectory.html")